In [27]:
import pandas as pd

In [28]:
df1 = pd.read_csv("to_be_final_crawling.csv")

In [29]:
df1.rename(columns = {'cat4': 'by_sort', 'cat2':'by_situation', 'cat3':'by_ingredient'}, inplace = True)

In [31]:
df1.head(3)

,index,name,best_name,rating,rating_count,author,author_type,totalTime,recipeYield,recipeLev,image,description,datePublished,recipeIngredient,recipeInstructions,by_sort,by_ingredient,by_situation
0,6996835,시래기 우거지 코다리찜 만드는법,시래기우거지코다리찜,0.0,0,Tinassoulfood,person,PT0H60M,4 servings,아무나,https://recipe1.ezmember.co.kr/cache/recipe/20...,시장에 갔다가 코다리를 사왔어요. 시래기나 우거지 넣고 끓인 코다리찜 은근 깊은 맛...,2023-02-11T12:20:13+09:00,코다리 4마리|냉동 시래기 1줌|냉동 우거지 2줌|양파 1개|무 4cm|멸치다시마육...,코다리는 잘 마른 걸 사면 그냥 사용해도 되지만 냉동 상태 코다리는 녹으면 생물 상...,메인반찬,건어물류,일상
1,6915124,와인과 잘 어울리는 시금치 치킨 소바~!,시금치치킨소바,0.0,0,칠갑농산,person,PT30M,2 servings,중급,https://recipe1.ezmember.co.kr/cache/recipe/20...,오늘은 메밀국수를 시금치와 닭고기를 곁들인 요리 어떠세요? 와인과도 아주 잘 어울리...,2019-07-02T13:21:57+09:00,칠갑농산 메밀국수 200g|닭가슴살 300g|다진 시금치 3컵|양파 1개|다진마늘 ...,"재료 (2인기준)입니다.|닭가슴살, 양파, 시금치를 잘게 다집니다.|팬에 기름을 두...",기타,닭고기,손님접대
2,6992711,두부 구이 어묵 볶음 밥반찬 도시락 간단레시피,두부구이,0.0,0,요리조이,person,PT15M,5 servings,초급,https://recipe1.ezmember.co.kr/cache/recipe/20...,바쁠때는 모두 볶아주세요,2022-11-26T12:00:08+09:00,두부 380g|어묵 2조각|대파 3조각|중멸치 조금|식용유 약간|설탕 1T|양조간장...,멸치의 내장을 제거합니다|두부는 반 잘라서 균일하게 썰어 주세요|어묵과 대파를 썰어...,밑반찬,콩/견과류,일상


1. index : 데이터 인덱스, 'https://www.10000recipe.com/recipe/{index}'-> 가면 레시피 상세정보 확인가능
2. name : 레시피 제목
3. best_name : 순수 레시피 제목
4. rating : 레시피 평점 평균
5. rating_count : 레시피 평점 갯수. 레시피 평점 총합 / 레시피 평점 갯수
6. author : 레시피 작성자
7. author_type : 레시피 작성자 타입. 근데 만들어 놓고 보니까 다 person임.
8. totalTime : 레시피 소요시간
9. recipeLev : 레시피 난이도
10. image : 이미지 url
11. description : 레시피 설명
12. dataPublished : 레시피 생성 날짜
13. recipeIngredient : 레시피 재료
14. recipeInstructions : 레시피 설명

In [14]:
df1 = df1.drop('recipeInstructions', axis=1) # 필요없는 레시피 설명 삭제.
df1 = df1.drop('author_type', axis=1)        # 필요없는 작성자 타입 삭제.

In [15]:
# image 분리 및 누락 값이 존재하는 데이터 삭제

df1[['image1', 'image2']] = df1['image'].str.split('|', expand=True)
df1 = df1.loc[~((df1['image1'] == 'https://recipe1.ezmember.co.kr/cache') | (df1['image2'] == 'https://recipe1.ezmember.co.kr/cache'))]

In [16]:
# 누락 값이 있는 행 삭제.

df1 = df1.loc[~(df1['author'].isna())]
df1 = df1.loc[~(df1['name'].isna())]
df1 = df1.loc[~(df1['totalTime'].isna())]
df1 = df1.loc[~(df1['recipeYield'].isna())]
df1 = df1.loc[~(df1['description'].isna())]

---------

In [17]:
df2 = pd.read_csv("ingredients_05_30.csv")
df2 = df2[df2['count'] >= 354]
name_list = df2['name'].tolist()

ingredient_df = pd.DataFrame({
    'ingredient_id': range(0, len(name_list)),
    'ingredient_name': [f'{i}' for i in name_list]
})


In [26]:
ingredient_df.to_csv("Last_Ingredients.csv", index=False)

In [18]:
temp = df1[df1['recipeIngredient'].str.contains('|'.join(name_list))]
temp.reset_index(inplace=True, drop=True)

In [23]:
from tqdm import tqdm

temp['Ingr1'] = ''
temp['Ingr2'] = ''
for i in tqdm(range(len(temp))):
    for j in range(len(name_list)):
        if name_list[j] in temp['recipeIngredient'][i]:
            temp.at[i, 'Ingr1'] += str(ingredient_df[ingredient_df['ingredient_name'] == name_list[j]]['ingredient_id'].values[0]) + ','
            temp.at[i, 'Ingr2'] += str(name_list[j]) + '|'

C:\Users\lopin\AppData\Local\Temp\ipykernel_8664\1985881143.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['Ingr1'] = ''
C:\Users\lopin\AppData\Local\Temp\ipykernel_8664\1985881143.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['Ingr2'] = ''
100%|██████████| 198242/198242 [09:54<00:00, 333.31it/s]


In [24]:
temp = temp.drop('image', axis=1)
temp.to_csv("Last_recipe.csv", index=False)

In [105]:
temp[['Ingr2', 'name', 'recipeIngredient']].head()

,Ingr2,name,recipeIngredient
0,소금|대파|간장|마늘|당근|고추|맛술|두부|된장|무|청양고추|김치|조각|홍고추|버섯...,시원한 배추김치국,김치 1/4포기|대파 1/2쪽|두부 1/2모|버섯 적당히|당근 작은거 1/2개|청양...
1,고추|계란|치즈|김치|김|청고추|,"[무드앤쿡#23.] 밥이랑 먹을까, 술이랑 먹을까?! 업그레이드 된 클래식 메뉴 <...",청고추 1개|김치 20g|치즈 기호에맞게|계란 5개
2,설탕|다진마늘|후추|마늘|생강|올리브유|김치|후추가루|김|배|배추|다진생강|배추김치...,돼지목살이 듬뿍들어간 김치찌개!,배추김치 1/4등분|돼지목살 500g|다진마늘 1큰술|다진생강 1티스푼|올리브유 1...
3,양파|소금|참기름|대파|후추|마늘|당근|호박|쌀|닭가슴살|닭|월계수잎|찹쌀|콩|,닭가슴살로 만드는 영양만점 닭죽만들기,닭가슴살|대파|통마늘|월계수잎|찹쌀|흰쌀|렌즈콩|호박|당근|양파|참기름|소금|후추
4,설탕|소금|계란|박력분|베|크림|커피|요거트|,노버터 커피 카라멜 파운드케이크,계란 2개|설탕 70g|소금 1꼬집|카라멜 크림 60g|오일 60g|가당 플레인 요...


----

핵심재료: 주된 맛과 영양을 제공하는 재료
부재료: 보조적인 맛과 질감을 제공하는 재료
소스류: 음식의 맛을 내거나 결합하는 데 사용되는 액체성 재료
향신료: 음식의 향미를 더하는 건조 또는 분말 형태의 재료
채소류: 주로 비타민, 미네랄 등의 영양을 제공하는 채소 재료
과일류: 단맛과 신선함을 더하는 과일 재료
곡물류: 전분과 섬유질을 제공하는 곡물 재료
유제품: 우유, 치즈, 버터 등의 유제품 재료
해산물: 생선, 조개 등의 해산물 재료

In [2]:
import pandas as pd

df2 = pd.read_csv("ingredients_05_30.csv")
df2 = df2[df2['count'] >= 504]

df2['ingredient_category'] = ''
for i, row in df2.iterrows():
    ingredient = row['name']
    print(f"'{ingredient}'은 어떤 카테고리에 속하나요?")
    category = input()
    df2.at[i, 'ingredient_category'] = category

'설탕'은 어떤 카테고리에 속하나요?


In [20]:
df2

,name,count
0,설탕,66336
1,양파,62812
2,소금,59010
3,참기름,51192
4,대파,45024
...,...,...
276,카레,514
277,두유,507
278,마늘가루,507
279,안심,507


In [7]:
import pandas as pd

df3 = pd.read_csv("0608/recipe_0608.csv")

count = 0
for i, row in df3.iterrows():
    if count == 10:
        break
    print(row['name'])
    print(row['Ingr2'])
    print(row['recipeIngredient'])
    print('\n')
    count += 1

시원한 배추김치국
소금|대파|간장|마늘|당근|고추|맛술|두부|된장|무|청양고추|김치|조각|홍고추|버섯|김|맛소금|간|미림|다시마|
김치 1/4포기|대파 1/2쪽|두부 1/2모|버섯 적당히|당근 작은거 1/2개|청양고추|홍고추 1줌|무 3조각 (1센치 두께)|다시마 1개 (손바닥 크기)|물 4~5컵|미림|맛술 2큰술|마늘 1큰술|된장 1큰술|맛소금 1/3큰술|간장 1/2큰술 (간 조절)


[무드앤쿡#23.] 밥이랑 먹을까, 술이랑 먹을까?! 업그레이드 된 클래식 메뉴 <김치치즈 계란말이> 
고추|계란|치즈|김치|김|청고추|
청고추 1개|김치 20g|치즈 기호에맞게|계란 5개


돼지목살이 듬뿍들어간 김치찌개!
설탕|다진마늘|후추|마늘|생강|올리브유|김치|후추가루|김|배|배추|다진생강|배추김치|목살|돼지|
배추김치 1/4등분|돼지목살 500g|다진마늘 1큰술|다진생강 1티스푼|올리브유 1큰술|후추가루 조금|설탕 2큰술|물 1L


닭가슴살로 만드는 영양만점 닭죽만들기
양파|소금|참기름|대파|후추|마늘|당근|호박|쌀|닭가슴살|닭|월계수잎|찹쌀|콩|
닭가슴살|대파|통마늘|월계수잎|찹쌀|흰쌀|렌즈콩|호박|당근|양파|참기름|소금|후추


노버터 커피 카라멜 파운드케이크 
설탕|소금|계란|박력분|베|크림|커피|요거트|
계란 2개|설탕 70g|소금 1꼬집|카라멜 크림 60g|오일 60g|가당 플레인 요거트 40g|인스턴트 커피 1ts|박력분 120g|베이킹 파우더 1ts|시나몬 파우더 1/4ts|구워서 반으로 쪼갠 헤이즐넛 30g


백종원부대찌개 레시피 간단하고 맛좋아서 추천해요
양파|대파|다진마늘|간장|후추|마늘|고춧가루|고추장|고추|국간장|치즈|된장|김치|콩나물|참치액|참치|김|스팸|간|소세지|신김치|콩|집된장|
스팸 1캔|콩나물 100g|후랑크소세지 3개|신김치 100g|양파 1/2개|대파 1줌|슬라이스치즈 1장|물 3컵|고춧가루 2T|고추장 0.5T|집된장 0.5T|국간장 1T|참치액 2T|다진마늘 1T|후추


식빵계란빵 만들기
설탕|버터|계란|식빵|파슬리|빵|